In [45]:
from enum import Enum, IntFlag
import hashlib
import re
from typing import Any, Self
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_serializer,
    field_validator,
    model_serializer,
    model_validator,
    SecretStr,
)

VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

Novamente temos a importação das bibliotecas usadas e a definição do modelo de nome e de senha desejados.

In [46]:
class Role(IntFlag):
    Aluno = 0
    Personal = 1
    Dono = 2

class Dificuldade(str, Enum):
    Iniciante = "iniciante"
    Intermediario = "intermediario"
    Avancado = "avancado"

class Treino(BaseModel):
    name: str = Field(min_length=3, examples=["Treino de Peito"])
    descricao_treino: str = Field(min_length=10)
    dificuldade: Dificuldade
    duracao_treino: int = Field(gt=0)
    criador_do_treino: EmailStr

    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("O name do treino tem que ter no mínimo 3 caracteres.")
        return v.title()

    @model_validator(mode="after")
    def validate_treino(self) -> Self:
        if self.dificuldade == Dificuldade.avancado and self.duracao_treino < 30:
            raise ValueError("Um treino de nível avançado precisa durar mais que 30 minutos.")
        return self

class User(BaseModel):
    name: str = Field(examples=["Lucas"])
    email: EmailStr = Field(
        examples=["lucashenrique@gmail.com"],
        description="O endereço de email do usuário",
        frozen=True,
    )
    senha: SecretStr = Field(
        examples=["Senha123"], description="A senha do usuário", exclude=True
    )
    tipo_usuario: Role = Field(
        description="O tipo de usuário",
        examples=[0, 1, 2],
        default=0,
        validate_default=True,
    )
    nivel_de_treinamento: Dificuldade = Field(examples=["avançado"],)

    @field_validator("name")
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "O name é inválido!"
            )
        return v

    @field_validator("tipo_usuario", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Tipo de usuário inválido, por favor use um desses: {', '.join([x.name for x in Role])}"
            )

    @model_validator(mode="before")
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "senha" not in v:
            raise ValueError("O name e senha são necessários")
        if v["name"].casefold() in v["senha"].casefold():
            raise ValueError("A senha não pode conter um name")
        if not VALID_PASSWORD_REGEX.match(v["senha"]):
            raise ValueError(
                "A senha é inválida!"
            )
        v["senha"] = hashlib.sha256(v["senha"].encode()).hexdigest()
        return v

    @model_validator(mode="after")
    def validate_user_post(self, v: Any) -> Self:
        if self.tipo_usuario == Role.Dono and self.name != "Lucas":
            raise ValueError("Somente Lucas pode ser o dono da academia!")
        return self

    @field_serializer("tipo_usuario", when_used="json")
    @classmethod
    def serialize_role(cls, v) -> str:
        return v.name

    @model_serializer(mode="wrap", when_used="json")
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {"name": self.name, "função": self.tipo_usuario.name}
        return serializer(self)

Aqui são criadas as classes Role, Dificuldade, Treino e User. Nelas são usadas os validadores, tanto model quanto field, para validar seus campos, como foi feito nos exemplos.

In [47]:
def main() -> None:
    data = {
        "name": "Lucas",
        "email": "lucassiqueira@gmail.com",
        "senha": "Password123",
        "tipo_usuario": "Personal",
        "nivel_de_treinamento": "intermediario",
    }
    user = User.model_validate(data)
    if user:
        print(
            "The serializer that returns a dict:",
            user.model_dump(),
            sep="\n",
            end="\n\n",
        )
        print(
            "The serializer that returns a JSON string:",
            user.model_dump(mode="json"),
            sep="\n",
            end="\n\n",
        )
        print(
            "The serializer that returns a json string, excluding the role:",
            user.model_dump(exclude=["tipo_usuario"], mode="json"),
            sep="\n",
            end="\n\n",
        )
        print("The serializer that encodes all values to a dict:", dict(user), sep="\n")


if __name__ == "__main__":
    main()

The serializer that returns a dict:
{'name': 'Lucas', 'email': 'lucassiqueira@gmail.com', 'tipo_usuario': <Role.Personal: 1>, 'nivel_de_treinamento': <Dificuldade.Intermediario: 'intermediario'>}

The serializer that returns a JSON string:
{'name': 'Lucas', 'função': 'Personal'}

The serializer that returns a json string, excluding the role:
{'name': 'Lucas', 'email': 'lucassiqueira@gmail.com', 'nivel_de_treinamento': 'intermediario'}

The serializer that encodes all values to a dict:
{'name': 'Lucas', 'email': 'lucassiqueira@gmail.com', 'senha': SecretStr('**********'), 'tipo_usuario': <Role.Personal: 1>, 'nivel_de_treinamento': <Dificuldade.Intermediario: 'intermediario'>}


A main foi feita com alguns casos de teste usando o mesmo modelo mostrado na aula, mas levemente adaptado ao meu código.